# C3 — Baseline e Regressão Logística

## Objetivo

Rodar o `DummyClassifier` (baseline) e a `LogisticRegression` sobre a partição persistida pelo Notebook 03, usando **somente treino e validação**.

Este notebook:

- carrega o artefato supervisionado canônico via `load_supervised_artifact`;
- recupera treino e validação via `split_partitions`;
- **não abre o teste**;
- não faz tuning, calibração nem escolha de limiar;
- registra métricas e predições de validação em `outputs/`.

> O limiar usado aqui (0.5) é apenas diagnóstico. A escolha definitiva pertence ao C5.

### Pré-requisito

O projeto deve estar instalado em modo editável a partir da raiz:

```bash
pip install -e .
```

## 1. Bootstrap — imports

Toda a configuração vem de `src.utils.config` e `configs/supervised.yaml`. Nenhum caminho absoluto, nenhuma feature hard-coded.

A raiz do projeto é derivada da localização do pacote `src` instalado em modo editável — sem busca por diretórios nem manipulação de `sys.path`.

In [2]:
from pathlib import Path
import itertools
import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(
        f"Não encontrei a raiz do projeto (com 'src/' e 'pyproject.toml') "
        f"a partir de: {atual}"
    )


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("ROOT_DIR:", ROOT_DIR)
print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())

ROOT_DIR: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
Python: 3.14.8
Sistema: Windows 11


In [3]:
from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import (
    load_supervised_artifact,
    split_partitions,
)
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas
from src.models.baseline import build_baseline_pipeline, evaluate_baseline
from src.models.logistic import build_logistic_pipeline

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())

ROOT_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
DATA_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data
RAW_DIR           = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\raw
PROCESSED_DIR     = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed
ANALYTICAL_DIR    = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical
TREINO_DIR        = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\treino
OUTPUTS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs
LOGS_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\logs
CONFIGS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs
SUPERVISED_YAML   = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integra

## 2. Carregar o artefato supervisionado canônico

A partição já foi criada e persistida pelo Notebook 03. Este notebook **reutiliza** a partição, não cria outra.

O `load_supervised_artifact` já valida, ao ler o Parquet:

- presença de todas as features, do alvo e da coluna de partição;
- alvo binário 0/1;
- presença das três partições (`treino`, `validacao`, `teste`).

Não é preciso repetir essas checagens no notebook — qualquer inconsistência levanta exceção ainda na leitura.

O tipo de coorte vem de `SRAG_COHORT_KIND` (padrão `provisional`), alinhado com a decisão de manter explícita a fonte durante o desenvolvimento.

In [4]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
)

# Nomes canônicos vêm do manifesto — nada hard-coded aqui.
PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

print("Tipo de coorte:", manifest["cohort_kind"])
print("Dataset:", manifest["dataset_path"])
print("SHA-256:", manifest["dataset_sha256"])
print("Linhas:", manifest["rows"])
print("Colunas:", manifest["columns"])
print("Seed:", manifest["seed"])
print("Teste aberto?", manifest["test_opened"])
print("Partições:", manifest["partition_counts"])
print("Coluna de partição:", PARTICAO)
print("Coluna alvo:", ALVO)
print("Quantidade de features:", len(FEATURES))

display(df.head(3))

Tipo de coorte: provisional
Dataset: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical\dataset_supervisionado.parquet
SHA-256: 2eeca68132017f3728459e35b50d5747d881436ac5cf1f4f1afaec25a6de23e8
Linhas: 3915132
Colunas: 36
Seed: 42
Teste aberto? False
Partições: {'treino': 2740587, 'teste': 587274, 'validacao': 587271}
Coluna de partição: particao
Coluna alvo: alvo_obito_srag
Quantidade de features: 31


,nu_notific,grupo_split,ano_internacao,alvo_obito_srag,particao,idade_anos,dias_sintomas_ate_internacao,sexo_categoria,cs_gestant,cs_raca,...,sind_down,hepatica,asma,diabetes,neurologic,pneumopati,imunodepre,renal,obesidade,out_morbi
0,31781201987679,31781201987679,2026,0,treino,0.083333,9.0,feminino,6,4,...,2,2,2,2,2,2,2,2,2,1
1,31773493207022,31773493207022,2026,0,treino,4.000000,1.0,feminino,6,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,31774542694487,31774542694487,2026,0,treino,0.030116,0.0,feminino,6,4,...,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN


## 3. Recuperar as partições via `split_partitions`

`split_partitions(df, manifest)` devolve apenas `(treino, validacao)` — o teste fica de fora por construção. O bloco `teste` permanece inacessível por este caminho, alinhado com a disciplina do gate `G9_holdout`.

In [5]:
treino, validacao = split_partitions(df, manifest)

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")

X_val = validacao[FEATURES].copy()
y_val = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)

resumo_particoes = pd.DataFrame(
    {
        "particao": ["treino", "validacao"],
        "n": [len(treino), len(validacao)],
        "positivos": [int(y_train.sum()), int(y_val.sum())],
        "prevalencia": [float(y_train.mean()), float(y_val.mean())],
    }
)

display(resumo_particoes)
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val  :", X_val.shape, "| y_val  :", y_val.shape)

# Contagem do teste: apenas len(), sem materializar o bloco.
n_teste = int((df[PARTICAO].astype(str) == "teste").sum())
print(f"Teste: {n_teste:,} linhas — mantido fechado.")

,particao,n,positivos,prevalencia
0,treino,2740587,599327,0.218686
1,validacao,587271,128428,0.218686


X_train: (2740587, 31) | y_train: (2740587,)
X_val  : (587271, 31) | y_val  : (587271,)
Teste: 587,274 linhas — mantido fechado.


## 4. Checagem de leakage

`check_leakage` lê `features.proibidas` do `supervised.yaml` e confirma que nenhuma delas está em `X`.

In [6]:
check_leakage(X_train)
check_leakage(X_val)

proibidas = set(C.cfg("features", "proibidas") or [])
assert not (set(X_train.columns) & proibidas)
assert not (set(X_val.columns) & proibidas)
print("Nenhuma coluna proibida presente em X_train ou X_val.")

[check_leakage] OK — 31 colunas, nenhuma proibida presente.
[check_leakage] OK — 31 colunas, nenhuma proibida presente.
Nenhuma coluna proibida presente em X_train ou X_val.


## 5. Baseline — `DummyClassifier`

Estratégia `prior` (default do YAML): prevê a prevalência do treino. É o piso mínimo que qualquer modelo real precisa superar.

In [7]:
estrategia_dummy = (
    C.cfg("models", "baseline", "strategy") or "prior"
)

metricas_dummy = evaluate_baseline(
    X_train, y_train, X_val, y_val, strategy=estrategia_dummy
)

display(pd.DataFrame([metricas_dummy]))

,strategy,auc,ap,pr_auc,brier,log_loss,f1,precision,recall,sensibilidade,especificidade,accuracy,limiar,n
0,prior,0.5,0.218686,0.218686,0.170862,0.52524,0.0,0.0,0.0,0.0,1.0,0.781314,0.5,587271.0


## 6. Regressão Logística

`C=1.0` e `lbfgs` são apenas um ponto de partida — nenhuma decisão de tuning é tomada aqui. O `class_weight` vem do YAML.

In [8]:
from typing import Literal, cast

log_cfg = C.cfg("models", "logistic") or {}

Solver = Literal[
    "lbfgs",
    "liblinear",
    "newton-cg",
    "newton-cholesky",
    "sag",
    "saga",
]
solver_configurado = log_cfg.get("solver", "lbfgs")
solvers_validos = {
    "lbfgs",
    "liblinear",
    "newton-cg",
    "newton-cholesky",
    "sag",
    "saga",
}
if solver_configurado not in solvers_validos:
    raise ValueError(f"Solver inválido na configuração: {solver_configurado}")

pipe_log = build_logistic_pipeline(
    C=1.0,
    solver=cast(Solver, solver_configurado),
    max_iter=int(log_cfg.get("max_iter", 1000)),
    class_weight=log_cfg.get("class_weight", "balanced"),
)

pipe_log.fit(X_train, y_train)
prob_val = pipe_log.predict_proba(X_val)[:, 1]

if not np.isfinite(prob_val).all():
    raise RuntimeError("Probabilidades não finitas na validação.")
if not ((prob_val >= 0) & (prob_val <= 1)).all():
    raise RuntimeError("Probabilidades fora de [0, 1] na validação.")

metricas_log = metricas_completas(y_val, prob_val, limiar=0.5)
metricas_log = {
    **metricas_log,
    "modelo": "logistica_C1",
}

display(pd.DataFrame([metricas_log]))

,auc,ap,pr_auc,brier,log_loss,f1,precision,recall,sensibilidade,especificidade,accuracy,limiar,n,modelo
0,0.775253,0.440765,0.440765,0.198373,0.568213,0.50561,0.375226,0.774862,0.774862,0.638881,0.668618,0.5,587271.0,logistica_C1


## 7. Comparação na validação

Comparação provisória entre o piso (`Dummy`) e a logística. Nenhum dos dois é finalista — essa decisão pertence ao C4/C5.

In [9]:
linha_dummy = {
    "modelo": "dummy_prior",
    "auc": metricas_dummy["auc"],
    "ap": metricas_dummy["ap"],
    "brier": metricas_dummy["brier"],
    "f1": metricas_dummy["f1"],
    "precision": metricas_dummy["precision"],
    "recall": metricas_dummy["recall"],
    "accuracy": metricas_dummy["accuracy"],
    "limiar_diagnostico": metricas_dummy["limiar"],
    "n_validacao": int(metricas_dummy["n"]),
}

linha_log = {
    "modelo": "logistica_C1",
    "auc": metricas_log["auc"],
    "ap": metricas_log["ap"],
    "brier": metricas_log["brier"],
    "f1": metricas_log["f1"],
    "precision": metricas_log["precision"],
    "recall": metricas_log["recall"],
    "accuracy": metricas_log["accuracy"],
    "limiar_diagnostico": metricas_log["limiar"],
    "n_validacao": int(metricas_log["n"]),
}

comparacao = pd.DataFrame([linha_dummy, linha_log])
comparacao = comparacao.sort_values("ap", ascending=False).reset_index(drop=True)

display(comparacao)

delta_ap = float(linha_log["ap"] - linha_dummy["ap"])
print(f"Δ AP (logística − dummy): {delta_ap:+.4f}")
print("Se Δ ≤ 0, o pipeline precisa ser revisto antes de avançar.")

,modelo,auc,ap,brier,f1,precision,recall,accuracy,limiar_diagnostico,n_validacao
0,logistica_C1,0.775253,0.440765,0.198373,0.50561,0.375226,0.774862,0.668618,0.5,587271
1,dummy_prior,0.500000,0.218686,0.170862,0.00000,0.000000,0.000000,0.781314,0.5,587271


Δ AP (logística − dummy): +0.2221
Se Δ ≤ 0, o pipeline precisa ser revisto antes de avançar.


## 8. Predições de validação

Construímos uma tabela de predições apenas para a validação. Nada do teste.

In [10]:
pred_val = pd.DataFrame(
    {
        "y_true": y_val.to_numpy(),
        "y_prob": prob_val,
        "modelo": "logistica_C1",
        "cohort_kind": manifest["cohort_kind"],
    }
)

pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)

display(pred_val.head())
print("Total de predições de validação:", len(pred_val))

assert len(pred_val) == len(y_val)
assert pred_val["y_prob"].between(0, 1).all()

,y_true,y_prob,modelo,cohort_kind,y_pred
0,0,0.136600,logistica_C1,provisional,0
1,0,0.117045,logistica_C1,provisional,0
2,0,0.093664,logistica_C1,provisional,0
3,0,0.140513,logistica_C1,provisional,0
4,0,0.039457,logistica_C1,provisional,0


Total de predições de validação: 587271


## 9. Persistência dos artefatos

Salvamos tabelas em `outputs/tables/` e predições em `outputs/predictions/`. Nada de modelo treinado como candidato final.

In [11]:
import json

timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo = f"C3_baseline_logistica_{manifest['cohort_kind']}_{timestamp}"

caminho_metricas = TABLES_DIR / f"{prefixo}_metricas.csv"
caminho_comparacao = TABLES_DIR / f"{prefixo}_comparacao.csv"
caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_validacao.parquet"
caminho_manifesto = TABLES_DIR / f"{prefixo}_manifesto.json"

comparacao.to_csv(caminho_comparacao, index=False, encoding="utf-8-sig")
pred_val.to_parquet(caminho_predicoes, index=False)

manifesto_c3 = {
    "cohort_kind": manifest["cohort_kind"],
    "dataset_path": manifest["dataset_path"],
    "dataset_sha256": manifest["dataset_sha256"],
    "feature_columns": FEATURES,
    "target_column": ALVO,
    "partition_column": PARTICAO,
    "n_treino": int(len(treino)),
    "n_validacao": int(len(validacao)),
    "n_teste_fechado": n_teste,
    "limiar_usado": 0.5,
    "limiar_e_diagnostico": True,
    "tuning_realizado": False,
    "calibracao_aplicada": False,
    "teste_aberto": False,
    "modelos": ["dummy_prior", "logistica_C1"],
    "artefatos": {
        "comparacao": str(caminho_comparacao),
        "predicoes_validacao": str(caminho_predicoes),
    },
}

caminho_manifesto.write_text(
    json.dumps(manifesto_c3, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

# Round-trip mínimo
comp_lido = pd.read_csv(caminho_comparacao, encoding="utf-8-sig")
pred_lido = pd.read_parquet(caminho_predicoes)

assert len(comp_lido) == len(comparacao)
assert len(pred_lido) == len(pred_val)
assert pred_lido["y_prob"].between(0, 1).all()

print("Comparação    :", caminho_comparacao)
print("Predições     :", caminho_predicoes)
print("Manifesto     :", caminho_manifesto)

Comparação    : C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\tables\C3_baseline_logistica_provisional_20261014T174055Z_comparacao.csv
Predições     : C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\predictions\C3_baseline_logistica_provisional_20261014T174055Z_predicoes_validacao.parquet
Manifesto     : C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\tables\C3_baseline_logistica_provisional_20261014T174055Z_manifesto.json


## 10. Registro da execução

Registramos a execução no `Run` para reprodutibilidade. Nenhum resultado do teste entra aqui.

In [12]:
with Run(
    modelo="C3_baseline_logistica",
    parametros={
        "cohort_kind": manifest["cohort_kind"],
        "estrategia_dummy": estrategia_dummy,
        "logistic_C": 1.0,
        "logistic_solver": log_cfg.get("solver", "lbfgs"),
        "limiar_diagnostico": 0.5,
        "tuning_realizado": False,
        "calibracao_aplicada": False,
        "teste_aberto": False,
    },
    dataset={
        "n_treino": int(len(treino)),
        "n_validacao": int(len(validacao)),
        "n_teste_fechado": n_teste,
    },
    features={
        "n_features": len(FEATURES),
        "usadas": FEATURES,
    },
    notas=(
        "C3 — baseline e logística na partição persistida. "
        "Limiar 0.5 apenas diagnóstico. Teste não consultado."
    ),
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(
        dummy_ap=float(metricas_dummy["ap"]),
        dummy_auc=float(metricas_dummy["auc"]),
        logistica_ap=float(metricas_log["ap"]),
        logistica_auc=float(metricas_log["auc"]),
        logistica_brier=float(metricas_log["brier"]),
    )

print("Run registrado em:", r.pasta)

[reproducibility] run registrado em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs\2026-10-14_144055_943343_C3_baseline_logistica
Run registrado em: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs\2026-10-14_144055_943343_C3_baseline_logistica


## 11. Conclusão — baseline vs logística

Checagens de encerramento. O teste permaneceu fechado do começo ao fim.

In [13]:
checklist = {
    "artefato_carregado": "df" in globals() and "manifest" in globals() and not df.empty,
    "particoes_recuperadas": "treino" in globals() and "validacao" in globals(),
    "teste_nao_materializado": "teste" not in globals(),
    "dummy_executado": "metricas_dummy" in globals(),
    "logistica_executada": "metricas_log" in globals(),
    "probabilidades_validas": bool(pred_val["y_prob"].between(0, 1).all()),
    "artefatos_reabertos": True,
    "limiar_nao_escolhido": True,
    "tuning_nao_realizado": True,
    "calibracao_nao_aplicada": True,
}

checklist_df = pd.DataFrame(
    {"item": list(checklist.keys()), "concluido": list(checklist.values())}
)
display(checklist_df)

if not all(checklist.values()):
    pendencias = [k for k, v in checklist.items() if not v]
    raise RuntimeError(f"C3 terminou com pendências: {pendencias}")

print("C3 concluído. Próximo: C4 — Boosting.")
print("Lembretes: teste fechado, limiar 0.5 apenas diagnóstico, nenhum tuning/calibração nesta etapa.")

,item,concluido
0,artefato_carregado,True
1,particoes_recuperadas,True
2,teste_nao_materializado,True
3,dummy_executado,True
4,logistica_executada,True
5,probabilidades_validas,True
6,artefatos_reabertos,True
7,limiar_nao_escolhido,True
8,tuning_nao_realizado,True
9,calibracao_nao_aplicada,True


C3 concluído. Próximo: C4 — Boosting.
Lembretes: teste fechado, limiar 0.5 apenas diagnóstico, nenhum tuning/calibração nesta etapa.
